# Watermark detectors: benchmark of the detectors on public community sets

Every watermark detector of `src/filters/watermark/` (convnext, efficientnet, laion, clip, yolo, ocr) scores the public sets of `src/filters/benchmark.py`
(`boomb0om`: boomb0om/watermarks-validation, 122 images, the ConvNeXt authors' evaluation set; `scenery`: qwertyforce/scenery_watermarks, 1,000 per class, real photographers' marks, but the training set of `efficientnet`; `wm_or_not`: prithivMLmods/Watermark-or-Not-20K, 1,000 per class, synthetic text marks). The report gives, per model and per set, accuracy, precision, recall, F1 and ROC-AUC at threshold
0.5, the latency of one image (batch 1, p50 and p95, after a warm-up) and the batched throughput; then the votes of
every subset of two or more models (mean, max, majority) and the choice between the best single model and the best
vote. Results: `out/filters/watermark/benchmark/evaluation.md` (tables for the report) and `evaluation.json`.

Speed and memory: both T4 GPUs work at once, one model per GPU, slow models first; every model runs in its own
process and its own Python environment (conflicting libraries never meet, and a model's memory is freed when it
ends); images of the next batches are decoded on the CPU while the GPU runs. Benchmark images stay in `/tmp`.

Before running: GPU T4 x2, internet on, Kaggle secret `HF_TOKEN`; the code is cloned from GitHub (`BRANCH`), so
push first; get the coordinator's approval. Time: about 30-60 min, of which 5-10 min to build the environments. `SMOKE = True` runs 20 images per model first.


In [ ]:
import os, shutil, subprocess, sys

REPO = "https://github.com/CryAndRRich/edit-judge.git"
BRANCH = "curr"
DATA = "/kaggle/input/editjudge-data"      # the private dataset (editjudge-data.zip, extracted by Kaggle)
CODE = "/kaggle/working/edit-judge"
OUT = "/kaggle/working/out"

if os.path.exists(CODE):
    shutil.rmtree(CODE)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, CODE], check=True)
print(subprocess.run(["git", "-C", CODE, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

os.environ["EJ_DATA_DIR"] = DATA           # read-only input
os.environ["EJ_OUTPUT_DIR"] = OUT          # all results are written here
os.makedirs(OUT, exist_ok=True)
os.chdir(CODE)

# Model weights go to /tmp: /kaggle/working is limited to 20 GB and is saved as the notebook's output
os.environ["HF_HOME"] = "/tmp/hf"
os.environ["EJ_MODELS_DIR"] = "/tmp/models"
os.environ["EJ_WORK_DIR"] = "/tmp/work"    # benchmark images and labels: never in the saved output
os.environ["EJ_ENVS_DIR"] = "/tmp/ej-envs" # one Python environment per group of conflicting packages
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
try:                                       # optional Kaggle secret HF_TOKEN (Add-ons > Secrets)
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:                          # public models need no token: hide the rate-limit notice
    os.environ["HF_HUB_VERBOSITY"] = "error"

PREVIOUS = None                            # e.g. "/kaggle/input/<this notebook's previous version>/out"
if PREVIOUS:
    shutil.copytree(f"{PREVIOUS}/filters", f"{OUT}/filters", dirs_exist_ok=True)

In [ ]:
!pip -q install uv                        # fast creation of the per-group environments (src/filters/envs.py)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -m src.filters list

import json
from IPython.display import Markdown, display

def ej(*args):
    """python -m src.filters <args>; prints and returns its output."""
    r = subprocess.run([sys.executable, "-m", "src.filters", *map(str, args)], capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-3000:])
    r.check_returncode()
    return r.stdout

def last_json(text):
    """The JSON object a command prints last (multirun prints progress lines before it)."""
    i = text.rfind("\n{")
    return json.loads(text[i + 1:] if i >= 0 else text)

def show_failures(result):
    for name, why in last_json(result)["failed"].items():
        print("FAILED", name, why)
        log = f"{OUT}/filters/logs/{name}.log"
        if os.path.exists(log):
            print(open(log).read()[-2000:])

In [ ]:
TASK = "watermark"
MODELS = None           # None = every registered detector, or a list such as ["convnext", "laion"]
SETS = None             # None = every public set of the task, or a list of set names
LATENCY = 100           # images timed one by one (batch 1) per model
SMOKE = False           # True: 20 images per model, to check that every model loads and runs


In [ ]:
ej("prepare", "--task", TASK, *(["--sets", ",".join(SETS)] if SETS else []))
args = ["multirun", "--task", TASK, "--on", "benchmark", "--latency", 5 if SMOKE else LATENCY]
args += ["--models", ",".join(MODELS)] if MODELS else []
args += ["--sets", ",".join(SETS)] if SETS else []
args += ["--limit", 20] if SMOKE else []
result = ej(*args)
show_failures(result)

In [ ]:
ej("evaluate", "--task", TASK, "--on", "benchmark")
display(Markdown(open(f"{OUT}/filters/{TASK}/benchmark/evaluation.md").read()))